
# DS5201 Deep Learning Project 2
## CNNs from Scratch vs Transfer Learning for Image Classification

This notebook implements all required tasks:
1. Dataset selection, exploration, preprocessing
2. Data augmentation
3. CNN from scratch
4. Transfer learning (frozen base)
5. Fine-tuning
6. Experimental comparison
7. Report-ready outputs (tables, plots, confusion matrix, misclassifications)



### Reproducibility and Runtime Notes
- Dataset: **CIFAR-100** (subset of classes configurable)
- Framework: **TensorFlow / Keras**
- Pretrained model: **MobileNetV2 (ImageNet weights)**
- This notebook is designed to be complete and modular.
- For fast experimentation, tune `MAX_TRAIN_SAMPLES` and `EPOCHS_*`.


In [ ]:

# Core imports
import os
import time
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)


In [ ]:

# -------------------------
# Global configuration
# -------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# Class subset (10 classes selected from CIFAR-100)
SELECTED_CLASS_INDICES = [0, 1, 7, 8, 12, 13, 14, 18, 19, 23]

# Train speed controls (set to None to use all selected data)
MAX_TRAIN_SAMPLES = None
MAX_VAL_SAMPLES = None
MAX_TEST_SAMPLES = None

# Training hyperparameters
BATCH_SIZE = 64
EPOCHS_SCRATCH = 30
EPOCHS_TRANSFER = 20
EPOCHS_FINETUNE = 15
LEARNING_RATE_BASE = 1e-3
LEARNING_RATE_FINETUNE = 1e-4   # 10x smaller than base
EARLY_STOPPING_PATIENCE = 5

# Image settings
IMG_SIZE = (96, 96)  # good for MobileNetV2
NUM_CHANNELS = 3


## Task 1: Dataset Selection, Exploration, and Preprocessing

In [ ]:

# Load CIFAR-100
(x_train_full, y_train_full), (x_test_full, y_test_full) = keras.datasets.cifar100.load_data(label_mode='fine')
y_train_full = y_train_full.squeeze()
y_test_full = y_test_full.squeeze()

# Combine then split 70/15/15
X_all = np.concatenate([x_train_full, x_test_full], axis=0)
y_all = np.concatenate([y_train_full, y_test_full], axis=0)

# Keep selected classes only
mask = np.isin(y_all, SELECTED_CLASS_INDICES)
X_all = X_all[mask]
y_all = y_all[mask]

# Remap class ids to 0..N-1
class_id_map = {orig: idx for idx, orig in enumerate(SELECTED_CLASS_INDICES)}
y_all = np.array([class_id_map[int(c)] for c in y_all])

# CIFAR-100 fine labels
fine_label_names = [
    'apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle',
    'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel',
    'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock', 'cloud',
    'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur', 'dolphin', 'elephant',
    'flatfish', 'forest', 'fox', 'girl', 'hamster', 'house', 'kangaroo', 'keyboard',
    'lamp', 'lawn_mower', 'leopard', 'lion', 'lizard', 'lobster', 'man', 'maple_tree',
    'motorcycle', 'mountain', 'mouse', 'mushroom', 'oak_tree', 'orange', 'orchid', 'otter',
    'palm_tree', 'pear', 'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine',
    'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose', 'sea', 'seal', 'shark',
    'shrew', 'skunk', 'skyscraper', 'snail', 'snake', 'spider', 'squirrel', 'streetcar',
    'sunflower', 'sweet_pepper', 'table', 'tank', 'telephone', 'television', 'tiger', 'tractor',
    'train', 'trout', 'tulip', 'turtle', 'wardrobe', 'whale', 'willow_tree', 'wolf', 'woman', 'worm'
]
selected_class_names = [fine_label_names[i] for i in SELECTED_CLASS_INDICES]

# Stratified split: 70/15/15
X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.30, random_state=SEED, stratify=y_all
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

# Optional subsampling for speed
if MAX_TRAIN_SAMPLES:
    idx = np.random.choice(len(X_train), min(MAX_TRAIN_SAMPLES, len(X_train)), replace=False)
    X_train, y_train = X_train[idx], y_train[idx]
if MAX_VAL_SAMPLES:
    idx = np.random.choice(len(X_val), min(MAX_VAL_SAMPLES, len(X_val)), replace=False)
    X_val, y_val = X_val[idx], y_val[idx]
if MAX_TEST_SAMPLES:
    idx = np.random.choice(len(X_test), min(MAX_TEST_SAMPLES, len(X_test)), replace=False)
    X_test, y_test = X_test[idx], y_test[idx]

print("Classes:", len(selected_class_names))
print("Class names:", selected_class_names)
print("Image dimensions:", X_train.shape[1:])
print("Train/Val/Test:", len(X_train), len(X_val), len(X_test))


In [ ]:
# Dataset statistics and class distribution

def class_distribution(y, class_names):
    counts = pd.Series(y).value_counts().sort_index()
    df = pd.DataFrame({
        'class_idx': counts.index,
        'class_name': [class_names[i] for i in counts.index],
        'count': counts.values
    })
    return df

train_dist = class_distribution(y_train, selected_class_names)
val_dist = class_distribution(y_val, selected_class_names)
test_dist = class_distribution(y_test, selected_class_names)

print('Train distribution:')
print(train_dist)
print()
print('Balanced check (train):')
print('min:', train_dist['count'].min(), 'max:', train_dist['count'].max())
imbalance_ratio = train_dist['count'].max() / train_dist['count'].min()
print('imbalance ratio (max/min):', round(float(imbalance_ratio), 3))
print('Assessment:', 'Balanced' if imbalance_ratio < 1.5 else 'Potentially imbalanced')

In [ ]:

# Visualize representative images from each class
plt.figure(figsize=(16, 8))
for i in range(len(selected_class_names)):
    class_idx = i
    sample_idx = np.where(y_train == class_idx)[0][0]
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[sample_idx])
    plt.title(selected_class_names[class_idx])
    plt.axis('off')
plt.suptitle('Representative Training Images per Class')
plt.tight_layout()
plt.show()


In [ ]:

# Resize + normalize for scratch model (0-1 scaling)

def preprocess_for_scratch(images):
    images = tf.image.resize(images, IMG_SIZE).numpy()
    images = images.astype('float32') / 255.0
    return images

X_train_s = preprocess_for_scratch(X_train)
X_val_s = preprocess_for_scratch(X_val)
X_test_s = preprocess_for_scratch(X_test)

num_classes = len(selected_class_names)

y_train_oh = keras.utils.to_categorical(y_train, num_classes)
y_val_oh = keras.utils.to_categorical(y_val, num_classes)
y_test_oh = keras.utils.to_categorical(y_test, num_classes)

print("Scratch-ready train shape:", X_train_s.shape)


## Task 2: Data Augmentation

In [ ]:

# Augmentation pipeline (applied only to training)
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.1),
], name='data_augmentation')

# Visualize augmentations
sample_images = X_train_s[:8]
augmented = data_augmentation(sample_images, training=True)

plt.figure(figsize=(16, 4))
for i in range(8):
    plt.subplot(2, 8, i + 1)
    plt.imshow(sample_images[i])
    plt.axis('off')
    if i == 0:
        plt.ylabel('Original', fontsize=10)

    plt.subplot(2, 8, i + 9)
    plt.imshow(tf.clip_by_value(augmented[i], 0, 1))
    plt.axis('off')
    if i == 0:
        plt.ylabel('Augmented', fontsize=10)

plt.suptitle('Original vs Augmented Images')
plt.tight_layout()
plt.show()


In [ ]:
# tf.data pipelines
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(X, y, train=False, preprocess_fn=None, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if train:
        ds = ds.shuffle(buffer_size=len(X), seed=SEED)
    ds = ds.batch(BATCH_SIZE)

    def _map_fn(x, t):
        if augment:
            x = data_augmentation(x, training=True)
        if preprocess_fn is not None:
            x = preprocess_fn(x)
        return x, t

    ds = ds.map(_map_fn, num_parallel_calls=AUTOTUNE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds_aug = make_ds(X_train_s, y_train_oh, train=True, augment=True)
train_ds_noaug = make_ds(X_train_s, y_train_oh, train=False, augment=False)
val_ds = make_ds(X_val_s, y_val_oh, train=False, augment=False)
test_ds = make_ds(X_test_s, y_test_oh, train=False, augment=False)


## Task 3: CNN from Scratch

In [ ]:

def build_scratch_cnn(input_shape, num_classes):
    model = keras.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),

        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ], name='scratch_cnn')
    return model

scratch_model = build_scratch_cnn((IMG_SIZE[0], IMG_SIZE[1], NUM_CHANNELS), num_classes)
scratch_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE_BASE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

scratch_model.summary()


In [ ]:
def make_callbacks():
    return [
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True
        )
    ]

start = time.time()
history_scratch = scratch_model.fit(
    train_ds_aug,
    validation_data=val_ds,
    epochs=EPOCHS_SCRATCH,
    callbacks=make_callbacks(),
    verbose=1
)
time_scratch = time.time() - start

print(f"Scratch training time: {time_scratch:.2f}s")


## Task 4: Transfer Learning (Frozen Base)

In [ ]:
# Preprocess for MobileNetV2
mobilenet_preprocess = tf.keras.applications.mobilenet_v2.preprocess_input

def resize_to_unit(images):
    images = tf.image.resize(images, IMG_SIZE).numpy().astype('float32')
    images = images / 255.0
    return images

def preprocess_for_mobilenet(images):
    images = resize_to_unit(images)
    images = mobilenet_preprocess(images * 255.0)
    return images

# Keep transfer inputs as resized [0, 1] tensors, augment, then convert to MobileNetV2 input space
X_train_t = resize_to_unit(X_train)
X_val_t = resize_to_unit(X_val)
X_test_t = resize_to_unit(X_test)

transfer_preprocess_fn = lambda x: mobilenet_preprocess(x * 255.0)

train_ds_t_aug = make_ds(
    X_train_t, y_train_oh, train=True, preprocess_fn=transfer_preprocess_fn, augment=True
)
val_ds_t = make_ds(
    X_val_t, y_val_oh, train=False, preprocess_fn=transfer_preprocess_fn, augment=False
)
test_ds_t = make_ds(
    X_test_t, y_test_oh, train=False, preprocess_fn=transfer_preprocess_fn, augment=False
)


In [ ]:

def build_transfer_model(input_shape, num_classes, base_trainable=False):
    base = tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    base.trainable = base_trainable

    inputs = keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='transfer_mobilenetv2')
    return model, base

transfer_model, transfer_base = build_transfer_model((IMG_SIZE[0], IMG_SIZE[1], NUM_CHANNELS), num_classes, base_trainable=False)
transfer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE_BASE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

transfer_model.summary()
print("Base trainable:", transfer_base.trainable)


In [ ]:

start = time.time()
history_transfer = transfer_model.fit(
    train_ds_t_aug,
    validation_data=val_ds_t,
    epochs=EPOCHS_TRANSFER,
    callbacks=make_callbacks(),
    verbose=1
)
time_transfer = time.time() - start

print(f"Transfer (frozen base) training time: {time_transfer:.2f}s")


## Task 5: Fine-Tuning

In [ ]:

# Unfreeze upper layers only
transfer_base.trainable = True

# Keep early layers frozen; unfreeze last ~40 layers
fine_tune_at = max(0, len(transfer_base.layers) - 40)
for layer in transfer_base.layers[:fine_tune_at]:
    layer.trainable = False

fine_tune_model = transfer_model  # continue from transfer weights
fine_tune_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE_FINETUNE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Total base layers:", len(transfer_base.layers))
print("Unfrozen layers in base:", sum(int(l.trainable) for l in transfer_base.layers))


In [ ]:
start = time.time()
start_epoch = len(history_transfer.history['loss'])
end_epoch = start_epoch + EPOCHS_FINETUNE
history_finetune = fine_tune_model.fit(
    train_ds_t_aug,
    validation_data=val_ds_t,
    epochs=end_epoch,
    initial_epoch=start_epoch,
    callbacks=make_callbacks(),
    verbose=1
)
time_finetune = time.time() - start

print(f"Fine-tuning time: {time_finetune:.2f}s")


## Task 6: Experimental Comparison and Evaluation

In [ ]:

def evaluate_model(model, X, y_true, preprocess='scratch', batch_size=128):
    y_true = np.asarray(y_true)

    if preprocess == 'scratch':
        X_p = preprocess_for_scratch(X)
    elif preprocess == 'mobilenet':
        X_p = preprocess_for_mobilenet(X)
    else:
        raise ValueError("Unknown preprocess type")

    loss, acc = model.evaluate(
        X_p, keras.utils.to_categorical(y_true, num_classes), batch_size=batch_size, verbose=0
    )
    probs = model.predict(X_p, batch_size=batch_size, verbose=0)
    y_pred = np.argmax(probs, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )

    return {
        'test_loss': float(loss),
        'test_accuracy': float(acc),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1),
        'y_pred': y_pred,
        'probs': probs,
    }

# Evaluate all models
results_scratch = evaluate_model(scratch_model, X_test, y_test, preprocess='scratch')
results_transfer = evaluate_model(transfer_model, X_test, y_test, preprocess='mobilenet')
results_finetune = evaluate_model(fine_tune_model, X_test, y_test, preprocess='mobilenet')


In [ ]:
def model_stats(model):
    total = model.count_params()
    trainable = np.sum([np.prod(v.shape) for v in model.trainable_weights])
    return int(total), int(trainable)

def best_val_acc(history):
    return float(np.max(history.history.get('val_accuracy', [np.nan])))

def epochs_completed(history):
    return len(history.history.get('loss', []))

scratch_total, scratch_trainable = model_stats(scratch_model)
transfer_total, transfer_trainable = model_stats(transfer_model)
finetune_total, finetune_trainable = model_stats(fine_tune_model)

epochs_transfer = epochs_completed(history_transfer)
epochs_finetune_only = epochs_completed(history_finetune)
epochs_finetune_total = epochs_transfer + epochs_finetune_only

comparison_df = pd.DataFrame([
    {
        'Model': 'CNN from Scratch',
        'Best Val Accuracy': best_val_acc(history_scratch),
        'Test Accuracy': results_scratch['test_accuracy'],
        'Test Loss': results_scratch['test_loss'],
        'Precision': results_scratch['precision'],
        'Recall': results_scratch['recall'],
        'F1-Score': results_scratch['f1_score'],
        'Total Parameters': scratch_total,
        'Trainable Parameters': scratch_trainable,
        'Epochs Completed': epochs_completed(history_scratch),
        'Training Time (s)': time_scratch,
    },
    {
        'Model': 'Frozen Pretrained',
        'Best Val Accuracy': best_val_acc(history_transfer),
        'Test Accuracy': results_transfer['test_accuracy'],
        'Test Loss': results_transfer['test_loss'],
        'Precision': results_transfer['precision'],
        'Recall': results_transfer['recall'],
        'F1-Score': results_transfer['f1_score'],
        'Total Parameters': transfer_total,
        'Trainable Parameters': transfer_trainable,
        'Epochs Completed': epochs_transfer,
        'Training Time (s)': time_transfer,
    },
    {
        'Model': 'Fine-Tuned Pretrained',
        'Best Val Accuracy': best_val_acc(history_finetune),
        'Test Accuracy': results_finetune['test_accuracy'],
        'Test Loss': results_finetune['test_loss'],
        'Precision': results_finetune['precision'],
        'Recall': results_finetune['recall'],
        'F1-Score': results_finetune['f1_score'],
        'Total Parameters': finetune_total,
        'Trainable Parameters': finetune_trainable,
        'Epochs Completed': epochs_finetune_total,
        'Training Time (s)': time_transfer + time_finetune,
    },
])

comparison_df


In [ ]:

# Plot training/validation curves

def plot_history(history, title_prefix):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history.history['accuracy'], label='train_acc')
    axes[0].plot(history.history['val_accuracy'], label='val_acc')
    axes[0].set_title(f'{title_prefix} Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()

    axes[1].plot(history.history['loss'], label='train_loss')
    axes[1].plot(history.history['val_loss'], label='val_loss')
    axes[1].set_title(f'{title_prefix} Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(history_scratch, 'Scratch CNN')
plot_history(history_transfer, 'Frozen Transfer')
plot_history(history_finetune, 'Fine-Tuned Transfer')


In [ ]:

# Metric comparison visualization
metrics_to_plot = ['Test Accuracy', 'Precision', 'Recall', 'F1-Score']
plot_df = comparison_df.set_index('Model')[metrics_to_plot]

plot_df.plot(kind='bar', figsize=(10, 5))
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.xticks(rotation=20)
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

comparison_df


## Task 7: Report-Ready Analysis Outputs

In [ ]:

# Select best model by test accuracy
all_results = {
    'CNN from Scratch': results_scratch,
    'Frozen Pretrained': results_transfer,
    'Fine-Tuned Pretrained': results_finetune,
}
best_model_name = max(all_results, key=lambda k: all_results[k]['test_accuracy'])
print('Best model:', best_model_name)

if best_model_name == 'CNN from Scratch':
    best_model = scratch_model
    best_preprocess = 'scratch'
elif best_model_name == 'Frozen Pretrained':
    best_model = transfer_model
    best_preprocess = 'mobilenet'
else:
    best_model = fine_tune_model
    best_preprocess = 'mobilenet'

if best_preprocess == 'scratch':
    X_test_best = preprocess_for_scratch(X_test)
else:
    X_test_best = preprocess_for_mobilenet(X_test)

best_probs = best_model.predict(X_test_best, verbose=0)
best_preds = np.argmax(best_probs, axis=1)


In [ ]:

# Confusion matrix + classification report for best model
cm = confusion_matrix(y_test, best_preds)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=selected_class_names, yticklabels=selected_class_names)
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print(classification_report(y_test, best_preds, target_names=selected_class_names, digits=4, zero_division=0))


In [ ]:

# Misclassified examples (at least 10)
mis_idx = np.where(best_preds != y_test)[0]
print('Total misclassified:', len(mis_idx))

n_show = min(12, len(mis_idx))
plt.figure(figsize=(16, 10))
for i, idx in enumerate(mis_idx[:n_show]):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_test[idx])
    true_label = selected_class_names[y_test[idx]]
    pred_label = selected_class_names[best_preds[idx]]
    conf = best_probs[idx][best_preds[idx]]
    plt.title(f'T: {true_label}\nP: {pred_label}\nConf: {conf:.2f}', fontsize=9)
    plt.axis('off')

plt.suptitle(f'Misclassified Test Images - {best_model_name}')
plt.tight_layout()
plt.show()



## Technical Report Template Sections (13 Required)
Use your notebook outputs above to complete the report:
1. Introduction
2. Dataset Description
3. Data Preprocessing and Augmentation
4. CNN Architecture from Scratch
5. Transfer Learning Model
6. Fine-Tuning Strategy
7. Experimental Results
8. Comparative Analysis
9. Confusion Matrix and Classification Report
10. Misclassified Images
11. Critical Reflection (400–500 words)
12. Conclusion
13. References


In [ ]:

# Save comparison table for report inclusion
comparison_df.to_csv('comparison_results.csv', index=False)
print('Saved: comparison_results.csv')
comparison_df
